
# 3. TABAKALI ÖRNEKLEME (STRATIFIED SAMPLING)

## 3.1. Orijinal Veri Dağılımının İncelenmesi

**Amaç:** Edge-IIoTset veri setindeki normal trafik ve saldırı kayıtlarının dağılımlarını inceleyerek temsil edici bir çalışma veri seti oluşturmak.

Örnekleme yöntemi olarak tabakalı rastgele örnekleme (Stratified Random Sampling) yaklaşımı değerlendirilecektir.

Bu aşamada yaklaşık 400.000 kayıtlık bir çalışma veri seti için oransal örnekleme planı hazırlanacaktır.


**Çalıştırma uyarısı:** 3.5 aşaması `EdgeIIoT_400000_korumali.csv` dosyasını oluşturur. Dosya zaten varsa mevcut kod, verinin yanlışlıkla üzerine yazılmasını önlemek için `FileExistsError` üretir. **Mevcut sonuçları korumak istiyorsanız notebook'u yeniden baştan çalıştırmanız gerekmez.**


### Bağımsız Çalıştırma İçin Kaynak Dosya Hazırlığı

**Amaç:** Örnekleme aşamasında kullanılan 24 orijinal CSV dosyasını, önceki notebook'un değişkenlerine ihtiyaç duymadan tanımlamak.

**Yöntem:** Proje klasöründeki ham CSV dosyaları bulunur; hazır ML/DNN dosyaları hariç tutulur ve dosya sayısı doğrulanır.

**Beklenen Çıktı:** `ham_dosyalar` değişkeni, örnekleme hücrelerinin kullanabileceği 24 orijinal CSV dosyasını içeren liste olarak hazırlanır.

**Not:** Bu kurulum hücresi yeni analiz yapmaz; orijinal 3. bölümün başka notebook'tan bağımsız açılabilmesini sağlar. Mevcut sonuçları değiştirmez.


In [ ]:
from pathlib import Path

# Örnekleme bölümünün önceki notebook'a bağımlılığını kaldırma
veri_klasoru = Path(
    r"E:\Data mining\Network_Intrusion_Detection\archive\Edge-IIoTset dataset"
)

if not veri_klasoru.is_dir():
    raise FileNotFoundError(f"Ham veri klasörü bulunamadı: {veri_klasoru}")

hazir_dosyalar = {
    "ML-EdgeIIoT-dataset.csv",
    "DNN-EdgeIIoT-dataset.csv"
}

ham_dosyalar = sorted(
    (dosya for dosya in veri_klasoru.rglob("*.csv")
     if dosya.name not in hazir_dosyalar),
    key=lambda dosya: dosya.name
)

assert len(ham_dosyalar) == 24, (
    f"24 orijinal CSV bekleniyordu, bulunan: {len(ham_dosyalar)}"
)

print("Örnekleme için hazır ham CSV dosyası:", len(ham_dosyalar))


In [1]:

import pandas as pd

# Önceki analizde belirlenen orijinal kayıt sayıları
kayit_sayilari = {
    "Backdoor_attack.csv": 24862,
    "DDoS_HTTP_Flood_attack.csv": 229022,
    "DDoS_ICMP_Flood_attack.csv": 2914354,
    "DDoS_TCP_SYN_Flood_attack.csv": 2020120,
    "DDoS_UDP_Flood_attack.csv": 3201626,
    "MITM_attack.csv": 1229,
    "OS_Fingerprinting_attack.csv": 1001,
    "Password_attack.csv": 1053385,
    "Port_Scanning_attack.csv": 22564,
    "Ransomware_attack.csv": 10925,
    "SQL_injection_attack.csv": 51203,
    "Uploading_attack.csv": 37634,
    "Vulnerability_scanner_attack.csv": 145869,
    "XSS_attack.csv": 15915,
    "Distance.csv": 1143540,
    "Flame_Sensor.csv": 1070196,
    "Heart_Rate.csv": 165319,
    "IR_Receiver.csv": 1307778,
    "Modbus.csv": 159502,
    "phValue.csv": 746908,
    "Soil_Moisture.csv": 1192777,
    "Sound_Sensor.csv": 1512883,
    "Temperature_and_Humidity.csv": 1615722,
    "Water_Level.csv": 2295288
}

# Verileri tabloya dönüştürme
ornekleme_plani = pd.DataFrame(
    kayit_sayilari.items(),
    columns=["Dosya Adı", "Kayıt Sayısı"]
)

# Trafik grubunu belirleme
ornekleme_plani["Trafik Grubu"] = (
    ornekleme_plani["Dosya Adı"].apply(
        lambda ad: "Saldırı"
        if ad.endswith("_attack.csv")
        else "Normal"
    )
)

# Toplam kayıt ve orijinal yüzdeler
toplam_kayit = ornekleme_plani["Kayıt Sayısı"].sum()

ornekleme_plani["Orijinal Oran (%)"] = (
    ornekleme_plani["Kayıt Sayısı"]
    / toplam_kayit * 100
).round(3)

# 400.000 kayıt için oransal örnekleme hesabı
hedef_kayit = 400000

ornekleme_plani["Tahmini Örnek Sayısı"] = (
    ornekleme_plani["Kayıt Sayısı"]
    / toplam_kayit * hedef_kayit
).round().astype(int)

print("Toplam CSV dosyası:", len(ornekleme_plani))
print("Orijinal toplam kayıt:", toplam_kayit)

print("\nNormal ve saldırı gruplarının dağılımı:")
display(
    ornekleme_plani.groupby("Trafik Grubu")[
        "Kayıt Sayısı"
    ].sum().to_frame()
)

print("\nDosya bazında örnekleme planı:")
display(
    ornekleme_plani.sort_values(
        "Kayıt Sayısı", ascending=False
    )
)

print("\nTahmini örnek sayısı:",
      ornekleme_plani["Tahmini Örnek Sayısı"].sum())


Toplam CSV dosyası: 24
Orijinal toplam kayıt: 20939622

Normal ve saldırı gruplarının dağılımı:


,Kayıt Sayısı
Trafik Grubu,
Normal,11209913
Saldırı,9729709



Dosya bazında örnekleme planı:


,Dosya Adı,Kayıt Sayısı,Trafik Grubu,Orijinal Oran (%),Tahmini Örnek Sayısı
4,DDoS_UDP_Flood_attack.csv,3201626,Saldırı,15.290,61159
2,DDoS_ICMP_Flood_attack.csv,2914354,Saldırı,13.918,55672
23,Water_Level.csv,2295288,Normal,10.961,43846
3,DDoS_TCP_SYN_Flood_attack.csv,2020120,Saldırı,9.647,38589
22,Temperature_and_Humidity.csv,1615722,Normal,7.716,30864
21,Sound_Sensor.csv,1512883,Normal,7.225,28900
17,IR_Receiver.csv,1307778,Normal,6.245,24982
20,Soil_Moisture.csv,1192777,Normal,5.696,22785
14,Distance.csv,1143540,Normal,5.461,21845
15,Flame_Sensor.csv,1070196,Normal,5.111,20443



Tahmini örnek sayısı: 399999



### 3.1. Orijinal Veri Dağılımı — Sonuç ve Yorum

Edge-IIoTset veri setindeki 24 orijinal CSV dosyasında toplam 20.939.622 kayıt bulunmaktadır.

Dosya gruplarına göre yapılan analiz sonucunda:

- Normal trafik: 11.209.913 kayıt (%53,53)
- Saldırı trafiği: 9.729.709 kayıt (%46,47)

Örnekleme için 400.000 kayıt hedeflenerek dosyaların orijinal veri setindeki oranlarına göre alınması gereken yaklaşık kayıt sayıları hesaplanmıştır.

Ancak oransal örnekleme sonucunda MITM ve OS Fingerprinting gibi az sayıda kayda sahip saldırı türlerinin çalışma veri setinde çok düşük sayıda temsil edileceği görülmüştür.

Bu durum, özellikle çok sınıflı saldırı sınıflandırması ve model değerlendirmesi açısından bir sınırlılık oluşturmaktadır.

Bu nedenle örnekleme işlemi uygulanmadan önce azınlık sınıflarının yeterli düzeyde temsil edilmesi ve veri setinin genel dağılımının korunması birlikte değerlendirilecektir.




### 3.2. Azınlık Sınıflarının Örnekleme Açısından İncelenmesi

**Amaç:**

Edge-IIoTset veri setinden seçilmesi planlanan 400.000 kayıt içerisinde, az sayıda örneğe sahip saldırı türlerinin ne ölçüde temsil edileceğini belirlemek.

**Yöntem:**

Orijinal CSV dosyalarının kayıt sayıları kullanılarak oransal örnekleme hesaplaması yapılacaktır.

Örnek sayıları, toplamın tam olarak 400.000 olmasını sağlayacak şekilde en büyük kalan yöntemiyle tam sayılara dönüştürülecektir.

Ön inceleme amacıyla, 100 kaydın altında temsil edilen dosyalar belirlenecektir. Bu eşik, kesin bir istatistiksel yeterlilik ölçütü değil, potansiyel örnekleme sorunlarını belirlemek için kullanılan bir kontrol değeridir.

**Beklenen Çıktı:**

- 400.000 kayıtlık oransal örnekleme planı
- Az temsil edilen saldırı dosyaları
- Her dosya için planlanan örnek sayısı

Bu aşamada henüz gerçek veri seçimi yapılmayacaktır.


In [8]:

import numpy as np
import pandas as pd

# Hedef çalışma veri seti büyüklüğü
hedef_kayit = 400000

# Orijinal toplam kayıt sayısını hesaplama
toplam_kayit = ornekleme_plani["Kayıt Sayısı"].sum()

# Oransal örnek sayılarının hesaplanması
tam_kotalar = (
    ornekleme_plani["Kayıt Sayısı"]
    / toplam_kayit * hedef_kayit
)

# İlk aşamada aşağı yuvarlama
taban_kotalar = np.floor(tam_kotalar).astype(int)

ornekleme_plani["Seçilecek Kayıt"] = taban_kotalar

# Toplamı tam 400.000'e tamamlama
kalan = (
    hedef_kayit
    - ornekleme_plani["Seçilecek Kayıt"].sum()
)

# Küsuratı en büyük olan dosyalara kalan kayıtları ekleme
kusuratlar = tam_kotalar - taban_kotalar

indeksler = kusuratlar.nlargest(kalan).index

ornekleme_plani.loc[
    indeksler, "Seçilecek Kayıt"
] += 1

# 100 kayıttan az temsil edilen dosyaları belirleme
az_temsil_edilenler = ornekleme_plani[
    ornekleme_plani["Seçilecek Kayıt"] < 100
].copy()

print("Orijinal toplam kayıt:", toplam_kayit)

print(
    "Planlanan toplam örnek:",
    ornekleme_plani["Seçilecek Kayıt"].sum()
)

print(
    "\n100 kayıttan az temsil edilen dosya sayısı:",
    len(az_temsil_edilenler)
)

print("\nAz temsil edilen dosyalar:")

display(
    az_temsil_edilenler[
        ["Dosya Adı", "Kayıt Sayısı", "Seçilecek Kayıt"]
    ]
)

print("\nNormal ve saldırı gruplarının örnek dağılımı:")

display(
    ornekleme_plani.groupby("Trafik Grubu")[
        "Seçilecek Kayıt"
    ].sum().to_frame()
)


Orijinal toplam kayıt: 20939622
Planlanan toplam örnek: 400000

100 kayıttan az temsil edilen dosya sayısı: 2

Az temsil edilen dosyalar:


,Dosya Adı,Kayıt Sayısı,Seçilecek Kayıt
5,MITM_attack.csv,1229,24
6,OS_Fingerprinting_attack.csv,1001,19



Normal ve saldırı gruplarının örnek dağılımı:


,Seçilecek Kayıt
Trafik Grubu,
Normal,214138
Saldırı,185862



### 3.2. Azınlık Sınıflarının Örnekleme Açısından İncelenmesi — Sonuç ve Yorum

Edge-IIoTset veri setindeki 20.939.622 kayıt üzerinden 400.000 kayıtlık oransal örnekleme planı hazırlanmıştır.

**Analiz Sonucunda Elde Edilen Bulgular:**

- Orijinal toplam kayıt sayısı: 20.939.622
- Planlanan örneklem büyüklüğü: 400.000
- Normal trafik örnek sayısı: 214.138 (%53,53)
- Saldırı trafiği örnek sayısı: 185.862 (%46,47)
- 100 kaydın altında temsil edilen dosya sayısı: 2

**Az Temsil Edilen Dosyalar:**

| Dosya Adı | Orijinal Kayıt | Planlanan Örnek |
|---|---:|---:|
| MITM_attack.csv | 1.229 | 24 |
| OS_Fingerprinting_attack.csv | 1.001 | 19 |

#### Bulguların Değerlendirilmesi

Oransal örnekleme yöntemi, dosya gruplarının orijinal veri setindeki yüzdelik dağılımlarını korumaktadır.

Ancak bu yöntem, az sayıda kayıt içeren saldırı türlerinin örneklemde yetersiz temsil edilmesine neden olabilmektedir.

Özellikle MITM ve OS Fingerprinting saldırılarının sırasıyla 24 ve 19 kayıt ile temsil edilmesi, çok sınıflı sınıflandırma modellerinin eğitilmesi ve değerlendirilmesi açısından bir sınırlılık oluşturabilir.

Bu nedenle yalnızca oransal örnekleme yönteminin kullanılması yerine, azınlık sınıflarının da dikkate alındığı bir tabakalı örnekleme yaklaşımının değerlendirilmesi gerekmektedir.

#### Sonuç

400.000 kayıtlık çalışma veri setinin oluşturulmasında normal ve saldırı trafiğinin genel dağılımının korunması önemlidir.

Bununla birlikte azınlık saldırı türlerinin yeterli temsil edilmesi de hedeflenmektedir.

Nihai örnekleme stratejisi belirlenmeden önce tüm orijinal verilerdeki gerçek saldırı sınıfı dağılımları analiz edilecektir.

Bu aşamada yalnızca örnekleme planı oluşturulmuş, henüz herhangi bir kayıt seçilmemiş veya değiştirilmemiştir.



### 3.3. Saldırı Sınıflarının Gerçek Dağılımının Analizi

**Amaç:**

Edge-IIoTset veri setindeki normal trafik ve saldırı türlerinin gerçek kayıt dağılımlarını belirlemek.

**Yöntem:**

24 orijinal CSV dosyasındaki `Attack_label` ve `Attack_type` sütunları parça parça okunacaktır.

Bellek kullanımını azaltmak amacıyla Pandas kütüphanesinin `chunksize` parametresi kullanılacaktır.

Her saldırı türünün toplam kayıt sayısı ve veri setindeki yüzdelik oranı hesaplanacaktır.

**Beklenen Çıktılar:**

- Normal ve saldırı sınıflarının gerçek kayıt sayıları
- Her saldırı türünün toplam veri setindeki oranı
- Azınlık ve çoğunluk sınıflarının belirlenmesi
- Temsil edici örnekleme stratejisi için istatistiksel temel

Bu aşamada veri temizleme, örnek seçme veya dosya birleştirme işlemi yapılmayacaktır.


In [9]:

import pandas as pd
from collections import Counter

# Saldırı sınıflarının kayıt sayılarını tutma
sinif_sayac = Counter()

# Etiket ve sınıf kombinasyonlarını tutma
etiket_sinif_sayac = Counter()

# 24 orijinal CSV dosyasını inceleme
for dosya in ham_dosyalar:

    # Belleği korumak için dosyaları parça parça okuma
    parcalar = pd.read_csv(
        dosya,
        usecols=["Attack_label", "Attack_type"],
        chunksize=50000,
        dtype=str
    )

    dosya_toplam = 0

    for parca in parcalar:

        # Etiketlerdeki boşlukları temizleme
        parca["Attack_label"] = (
            parca["Attack_label"]
            .fillna("(Eksik)")
            .str.strip()
        )

        parca["Attack_type"] = (
            parca["Attack_type"]
            .fillna("(Eksik)")
            .str.strip()
        )

        # Saldırı türlerinin kayıt sayılarını hesaplama
        sinif_sayac.update(
            parca["Attack_type"].value_counts().to_dict()
        )

        # Binary etiket ve saldırı türünü birlikte kontrol etme
        kombinasyonlar = (
            parca.groupby(
                ["Attack_label", "Attack_type"]
            ).size()
        )

        etiket_sinif_sayac.update(
            kombinasyonlar.to_dict()
        )

        dosya_toplam += len(parca)

    print(f"{dosya.name}: {dosya_toplam:,} kayıt")

# Sonuç tablosunu hazırlama
sinif_dagilimi = pd.DataFrame(
    sinif_sayac.items(),
    columns=["Saldırı Türü", "Kayıt Sayısı"]
)

toplam = sinif_dagilimi["Kayıt Sayısı"].sum()

sinif_dagilimi["Oran (%)"] = (
    sinif_dagilimi["Kayıt Sayısı"]
    / toplam * 100
).round(4)

sinif_dagilimi = (
    sinif_dagilimi.sort_values(
        "Kayıt Sayısı",
        ascending=False
    ).reset_index(drop=True)
)

print("\nToplam kayıt sayısı:", toplam)
print("Farklı sınıf sayısı:", len(sinif_dagilimi))

print("\nSaldırı sınıflarının gerçek dağılımı:")
display(sinif_dagilimi)

# Binary etiketler ile saldırı türlerini karşılaştırma
etiket_kontrolu = pd.DataFrame(
    [
        {
            "Attack_label": etiket,
            "Attack_type": tur,
            "Kayıt Sayısı": sayi
        }
        for (etiket, tur), sayi
        in etiket_sinif_sayac.items()
    ]
)

print("\nEtiket ve saldırı türü tutarlılık kontrolü:")
display(
    etiket_kontrolu.sort_values(
        "Kayıt Sayısı",
        ascending=False
    ).reset_index(drop=True)
)


Backdoor_attack.csv: 24,862 kayıt
DDoS_HTTP_Flood_attack.csv: 229,022 kayıt
DDoS_ICMP_Flood_attack.csv: 2,914,354 kayıt
DDoS_TCP_SYN_Flood_attack.csv: 2,020,120 kayıt
DDoS_UDP_Flood_attack.csv: 3,201,626 kayıt
Distance.csv: 1,143,540 kayıt
Flame_Sensor.csv: 1,070,196 kayıt
Heart_Rate.csv: 165,319 kayıt
IR_Receiver.csv: 1,307,778 kayıt
MITM_attack.csv: 1,229 kayıt
Modbus.csv: 159,502 kayıt
OS_Fingerprinting_attack.csv: 1,001 kayıt
Password_attack.csv: 1,053,385 kayıt
Port_Scanning_attack.csv: 22,564 kayıt
Ransomware_attack.csv: 10,925 kayıt
SQL_injection_attack.csv: 51,203 kayıt
Soil_Moisture.csv: 1,192,777 kayıt
Sound_Sensor.csv: 1,512,883 kayıt
Temperature_and_Humidity.csv: 1,615,722 kayıt
Uploading_attack.csv: 37,634 kayıt
Vulnerability_scanner_attack.csv: 145,869 kayıt
Water_Level.csv: 2,295,288 kayıt
XSS_attack.csv: 15,915 kayıt
phValue.csv: 746,908 kayıt

Toplam kayıt sayısı: 20939622
Farklı sınıf sayısı: 15

Saldırı sınıflarının gerçek dağılımı:


,Saldırı Türü,Kayıt Sayısı,Oran (%)
0,Normal,11209913,53.5345
1,DDoS_UDP,3201626,15.2898
2,DDoS_ICMP,2914354,13.9179
3,DDoS_TCP,2020120,9.6474
4,Password,1053385,5.0306
5,DDoS_HTTP,229022,1.0937
6,Vulnerability_scanner,145869,0.6966
7,SQL_injection,51203,0.2445
8,Uploading,37634,0.1797
9,Backdoor,24862,0.1187



Etiket ve saldırı türü tutarlılık kontrolü:


,Attack_label,Attack_type,Kayıt Sayısı
0,0,Normal,11209913
1,1,DDoS_UDP,3201626
2,1,DDoS_ICMP,2914354
3,1,DDoS_TCP,2020120
4,1,Password,1053385
5,1,DDoS_HTTP,229022
6,1,Vulnerability_scanner,145869
7,1,SQL_injection,51203
8,1,Uploading,37634
9,1,Backdoor,24862



### 3.3. Saldırı Sınıflarının Gerçek Dağılımının Analizi — Sonuç ve Yorum

Edge-IIoTset veri setinde bulunan 24 orijinal CSV dosyasındaki toplam 20.939.622 kaydın `Attack_label` ve `Attack_type` değişkenleri analiz edilmiştir.

**Analiz Sonucunda Elde Edilen Bulgular:**

- Toplam kayıt sayısı: 20.939.622
- Normal trafik kayıt sayısı: 11.209.913 (%53,5345)
- Saldırı trafiği kayıt sayısı: 9.729.709 (%46,4655)
- Toplam sınıf sayısı: 15
- Saldırı sınıfı sayısı: 14
- Normal trafik sınıfı sayısı: 1

**En Fazla Kayıt İçeren Sınıflar:**

| Sınıf | Kayıt Sayısı | Oran (%) |
|---|---:|---:|
| Normal | 11.209.913 | 53,5345 |
| DDoS_UDP | 3.201.626 | 15,2898 |
| DDoS_ICMP | 2.914.354 | 13,9179 |
| DDoS_TCP | 2.020.120 | 9,6474 |
| Password | 1.053.385 | 5,0306 |

**En Az Kayıt İçeren Sınıflar:**

| Sınıf | Kayıt Sayısı | Oran (%) |
|---|---:|---:|
| OS_Fingerprinting | 1.001 | 0,0048 |
| MITM | 1.229 | 0,0059 |
| Ransomware | 10.925 | 0,0522 |

#### Sınıf Dengesizliğinin Değerlendirilmesi

Analiz sonucunda saldırı sınıfları arasında belirgin bir veri dengesizliği (Class Imbalance) bulunduğu belirlenmiştir.

DDoS saldırı türleri milyonlarca kayıt içerirken MITM ve OS Fingerprinting gibi saldırı türleri oldukça az sayıda kayıtla temsil edilmektedir.

Bu durum, makine öğrenmesi modellerinin çoğunluk sınıflarını daha kolay öğrenmesine ve azınlık sınıflarındaki performanslarının düşük kalmasına neden olabilir.

Bu nedenle model değerlendirmesinde yalnızca Accuracy metriğinin kullanılması yeterli görülmemektedir.

Precision, Recall, Macro-F1 Score ve Confusion Matrix gibi ölçütlerin de değerlendirilmesi planlanmaktadır.

#### Saldırı Etiketlerinin Tutarlılığı

Tüm kayıtların `Attack_label` ve `Attack_type` değişkenleri birlikte kontrol edilmiştir.

Analiz sonucunda `Attack_label = 0` olan kayıtların Normal sınıfında, `Attack_label = 1` olan kayıtların ise saldırı sınıflarında bulunduğu doğrulanmıştır.

İncelenen etiket kombinasyonlarında çelişkili veya eksik etiket tespit edilmemiştir.

Bu bulgu, hedef değişkenlerin sınıflandırma çalışması için tutarlı bir etiket yapısına sahip olduğunu göstermektedir.

#### Örnekleme Açısından Değerlendirme

400.000 kayıtlık çalışma veri seti oluşturulurken yalnızca oransal örnekleme kullanılması, azınlık saldırı türlerinin çok düşük sayıda temsil edilmesine neden olabilir.

Bu nedenle nihai örnekleme stratejisinde hem orijinal sınıf dağılımı hem de azınlık sınıflarının yeterli düzeyde temsil edilmesi dikkate alınacaktır.

Örnekleme sonucunda oluşabilecek sınıf dağılımı değişiklikleri raporda açıkça belirtilecektir.

#### Sonuç

Edge-IIoTset veri setinde 14 saldırı türü ve 1 normal trafik sınıfı olmak üzere toplam 15 sınıf bulunduğu doğrulanmıştır.

Tüm kayıtlar üzerinde yapılan etiket analizi, veri setinde ciddi sınıf dengesizliği olduğunu ortaya koymuştur.

Elde edilen bulgular, temsil edici örnekleme stratejisinin belirlenmesi ve güvenilir sınıflandırma modellerinin geliştirilmesi için temel oluşturmaktadır.

Bu aşamada herhangi bir kayıt silinmemiş, değiştirilmemiş veya örneklenmemiştir.



### 3.4. Temsil Edici Örnekleme Stratejisinin Belirlenmesi

**Amaç:**

Edge-IIoTset veri setinden oluşturulacak 400.000 kayıtlık çalışma alt kümesi için uygun örnekleme stratejisini belirlemek.

**Yöntem:**

İki farklı örnekleme yaklaşımı karşılaştırılacaktır:

1. **Oransal Tabakalı Örnekleme:** Her dosyanın orijinal veri setindeki oranı korunarak kayıt sayısı belirlenir.

2. **Azınlık Sınıfı Korumalı Tabakalı Örnekleme:** Oransal örnekleme ile 300 kaydın altında kalan saldırı sınıflarına en az 300 kayıt ayrılır. Kalan örnekleme bütçesi diğer dosyalara oransal olarak dağıtılır.

Her iki yöntemde de toplam hedef kayıt sayısı 400.000 olarak belirlenmiştir.

300 kayıt sınırı, azınlık sınıflarının daha iyi temsil edilmesini incelemek amacıyla kullanılan deneysel bir eşiktir.

**Beklenen Çıktılar:**

- İki örnekleme yönteminin karşılaştırılması
- Azınlık saldırı türleri için örnek sayılarının belirlenmesi
- Normal ve saldırı trafiğinin yeni dağılımının hesaplanması
- Örnekleme stratejisinin avantaj ve sınırlılıklarının değerlendirilmesi

Bu aşamada henüz gerçek veri seçimi yapılmayacaktır.


In [10]:

import pandas as pd
import numpy as np

# Önceki analiz sonuçlarından yeni plan oluşturma
plan_34 = ornekleme_plani[
    ["Dosya Adı", "Kayıt Sayısı", "Trafik Grubu"]
].copy()

# Örnekleme parametreleri
hedef = 400000
minimum_saldiri = 300

# Birinci yöntem: Oransal örnekleme
oransal = (
    plan_34["Kayıt Sayısı"]
    / plan_34["Kayıt Sayısı"].sum()
    * hedef
)

oransal_taban = np.floor(oransal).astype(int)

plan_34["Oransal Kota"] = oransal_taban

# Yuvarlamadan kalan kayıtları dağıtma
kalan = hedef - plan_34["Oransal Kota"].sum()

kesirler = oransal - oransal_taban

indeksler = kesirler.nlargest(kalan).index

plan_34.loc[indeksler, "Oransal Kota"] += 1

# İkinci yöntem: Azınlık sınıflarını koruma
az_sinif = (
    (plan_34["Trafik Grubu"] == "Saldırı")
    & (plan_34["Oransal Kota"] < minimum_saldiri)
)

# Azınlık sınıfları için minimum kayıt sayısı
plan_34["Korunan Minimum"] = np.where(
    az_sinif,
    np.minimum(
        minimum_saldiri,
        plan_34["Kayıt Sayısı"]
    ),
    0
)

# Geriye kalan kayıt bütçesi
kalan_butce = (
    hedef - plan_34["Korunan Minimum"].sum()
)

diger_dosyalar = ~az_sinif

# Diğer dosyalara oransal dağıtım
yeni_kotalar = plan_34["Korunan Minimum"].astype(float)

yeni_kotalar.loc[diger_dosyalar] = (
    plan_34.loc[diger_dosyalar, "Kayıt Sayısı"]
    / plan_34.loc[diger_dosyalar, "Kayıt Sayısı"].sum()
    * kalan_butce
)

# Tam sayıya dönüştürme
yeni_taban = np.floor(yeni_kotalar).astype(int)

plan_34["Korumalı Kota"] = yeni_taban

# Toplamı 400.000'e tamamlama
kalan = hedef - plan_34["Korumalı Kota"].sum()

kesirler = yeni_kotalar - yeni_taban

indeksler = kesirler.nlargest(kalan).index

plan_34.loc[indeksler, "Korumalı Kota"] += 1

# İki yöntem arasındaki fark
plan_34["Kota Farkı"] = (
    plan_34["Korumalı Kota"]
    - plan_34["Oransal Kota"]
)

# Sonuçları gösterme
print("ORANSAL ÖRNEKLEME TOPLAMI:")
print(plan_34["Oransal Kota"].sum())

print("\nAZINLIK KORUMALI ÖRNEKLEME TOPLAMI:")
print(plan_34["Korumalı Kota"].sum())

print("\nÖRNEKLEME YÖNTEMLERİNİN KARŞILAŞTIRILMASI:")

display(
    plan_34[
        [
            "Dosya Adı",
            "Kayıt Sayısı",
            "Oransal Kota",
            "Korumalı Kota",
            "Kota Farkı"
        ]
    ]
)

print("\nNORMAL VE SALDIRI DAĞILIMI:")

display(
    plan_34.groupby("Trafik Grubu")[
        ["Oransal Kota", "Korumalı Kota"]
    ].sum()
)

print("\nAZINLIK SINIFLARININ KARŞILAŞTIRILMASI:")

display(
    plan_34.loc[
        az_sinif,
        [
            "Dosya Adı",
            "Oransal Kota",
            "Korumalı Kota"
        ]
    ]
)


ORANSAL ÖRNEKLEME TOPLAMI:
400000

AZINLIK KORUMALI ÖRNEKLEME TOPLAMI:
400000

ÖRNEKLEME YÖNTEMLERİNİN KARŞILAŞTIRILMASI:


,Dosya Adı,Kayıt Sayısı,Oransal Kota,Korumalı Kota,Kota Farkı
0,Backdoor_attack.csv,24862,475,474,-1
1,DDoS_HTTP_Flood_attack.csv,229022,4375,4368,-7
2,DDoS_ICMP_Flood_attack.csv,2914354,55672,55581,-91
3,DDoS_TCP_SYN_Flood_attack.csv,2020120,38589,38527,-62
4,DDoS_UDP_Flood_attack.csv,3201626,61159,61060,-99
5,MITM_attack.csv,1229,24,300,276
6,OS_Fingerprinting_attack.csv,1001,19,300,281
7,Password_attack.csv,1053385,20122,20090,-32
8,Port_Scanning_attack.csv,22564,431,430,-1
9,Ransomware_attack.csv,10925,209,300,91



NORMAL VE SALDIRI DAĞILIMI:


,Oransal Kota,Korumalı Kota
Trafik Grubu,,
Normal,214138,213790
Saldırı,185862,186210



AZINLIK SINIFLARININ KARŞILAŞTIRILMASI:


,Dosya Adı,Oransal Kota,Korumalı Kota
5,MITM_attack.csv,24,300
6,OS_Fingerprinting_attack.csv,19,300
9,Ransomware_attack.csv,209,300



### 3.4. Temsil Edici Örnekleme Stratejisinin Belirlenmesi — Sonuç ve Yorum

Edge-IIoTset veri setinden 400.000 kayıtlık bir çalışma alt kümesi oluşturmak amacıyla iki farklı tabakalı örnekleme stratejisi karşılaştırılmıştır.

**Karşılaştırılan Yöntemler:**

1. Oransal Tabakalı Örnekleme
2. Azınlık Sınıfı Korumalı Tabakalı Örnekleme

Her iki yöntemde de toplam örnek sayısı 400.000 olarak hesaplanmıştır.

#### 1. Oransal Örnekleme Sonuçları

Oransal örnekleme yöntemiyle:

- Normal trafik: 214.138 kayıt
- Saldırı trafiği: 185.862 kayıt
- Toplam: 400.000 kayıt

elde edilmesi planlanmıştır.

Bu yöntem, orijinal veri setindeki sınıf oranlarını korumaya yöneliktir.

Ancak azınlık saldırı sınıflarının çok düşük sayıda temsil edildiği görülmüştür.

#### 2. Azınlık Sınıfı Korumalı Örnekleme Sonuçları

Azınlık sınıflarının temsilini artırmak amacıyla oransal örnekleme sonucunda 300 kaydın altında kalan saldırı sınıflarına minimum 300 kayıt ayrılmıştır.

| Saldırı Türü | Oransal Örnek | Korumalı Örnek |
|---|---:|---:|
| MITM | 24 | 300 |
| OS_Fingerprinting | 19 | 300 |
| Ransomware | 209 | 300 |

Bu yöntemde:

- Normal trafik: 213.790 kayıt
- Saldırı trafiği: 186.210 kayıt
- Toplam: 400.000 kayıt

olarak hesaplanmıştır.

#### 3. Yöntemlerin Değerlendirilmesi

Oransal örnekleme, veri setinin orijinal sınıf dağılımını koruma açısından avantaj sağlamaktadır.

Bununla birlikte MITM ve OS Fingerprinting gibi azınlık sınıflarının çok az sayıda örnekle temsil edilmesi, sınıflandırma modellerinin eğitilmesi ve değerlendirilmesi açısından sınırlılıklar oluşturabilir.

Azınlık sınıfı korumalı örnekleme ise bu saldırı türlerinin çalışma veri setindeki örnek sayısını artırmaktadır.

Ancak bu yaklaşım, azınlık sınıflarının orijinal veri setindeki oranlarını değiştirdiği için tamamen oransal ve temsil edici bir örneklem olarak değerlendirilmemelidir.

Bu nedenle model değerlendirmesinde örnekleme kaynaklı dağılım değişikliklerinin dikkate alınması gerekmektedir.

#### 4. Sonuç

Yapılan karşılaştırma sonucunda her iki örnekleme yönteminin farklı avantajlara sahip olduğu belirlenmiştir.

Makine öğrenmesi uygulamalarında azınlık saldırı türlerinin daha iyi incelenebilmesi amacıyla azınlık sınıfı korumalı örnekleme yöntemi çalışma veri seti için aday yöntem olarak belirlenmiştir.

Bununla birlikte modelin gerçek veri dağılımındaki başarısının değerlendirilmesi için orijinal sınıf oranlarının korunması ve gerektiğinde örnekleme ağırlıklarının kullanılması ayrıca dikkate alınacaktır.

Örnekleme sırasında rastgele seçim yapılması, işlemin tekrarlanabilir olması ve eğitim-test verileri arasında veri sızıntısının önlenmesi hedeflenmektedir.

Bu aşamada yalnızca örnekleme kotaları hesaplanmış, gerçek kayıt seçimi henüz gerçekleştirilmemiştir.



### 3.5. Çalışma Veri Setinin Oluşturulması

**Amaç:**

Edge-IIoTset veri setindeki 20.939.622 kayıt arasından, farklı saldırı türlerini ve normal ağ trafiğini içeren 400.000 kayıtlık bir çalışma veri seti oluşturmak.

**Yöntem:**

Önceki aşamada hazırlanan azınlık sınıfı korumalı tabakalı örnekleme planı kullanılacaktır.

Her CSV dosyasından belirlenen kota kadar kayıt, sabit bir rastgelelik tohumu (`random_state=42`) kullanılarak tekrar seçimsiz biçimde rastgele alınacaktır.

Büyük dosyaların tamamını aynı anda belleğe yüklememek için veriler parçalar halinde okunacaktır.

Seçilen kayıtlar, orijinal veriler değiştirilmeden yeni bir CSV dosyasında birleştirilecektir.

**Kontrol Edilecek Özellikler:**

- Toplam kayıt sayısının 400.000 olması
- Her dosyadan belirlenen kota kadar kayıt seçilmesi
- 63 sütunun korunması
- Normal trafik ve saldırı türlerinin dağılımı
- Örnekleme işleminin tekrarlanabilir olması

**Sınırlılık:**

Azınlık sınıflarına daha yüksek kota ayrılması, orijinal sınıf oranlarında değişikliğe neden olmaktadır. Bu nedenle oluşturulan alt küme, orijinal dağılımın tamamen oransal bir temsili olarak değerlendirilmemelidir.

Model değerlendirmesinde bu dağılım değişikliği ve eğitim-test ayrımında veri sızıntısı riski ayrıca dikkate alınacaktır.


In [11]:

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

# Çalışma veri setinin kaydedileceği klasör
cikti_klasoru = Path(
    r"E:\Data mining\Network_Intrusion_Detection"
) / "Calisma_Verileri"

cikti_klasoru.mkdir(parents=True, exist_ok=True)

# Yeni veri setinin dosya adı
cikti_dosyasi = (
    cikti_klasoru / "EdgeIIoT_400000_korumali.csv"
)

# İşlem sırasında kullanılacak geçici dosya
gecici_dosya = (
    cikti_klasoru / "EdgeIIoT_400000_korumali.partial.csv"
)

# Mevcut dosyaların yanlışlıkla üzerine yazılmasını önleme
if cikti_dosyasi.exists():
    raise FileExistsError(
        f"Çıktı dosyası zaten mevcut: {cikti_dosyasi}"
    )

if gecici_dosya.exists():
    raise FileExistsError(
        f"Önceki işlemden geçici dosya kalmış: {gecici_dosya}"
    )

# Önceki aşamada hesaplanan kotaları alma
kota_tablosu = plan_34.set_index("Dosya Adı")

# Dosya ve kota kontrolleri
assert len(ham_dosyalar) == 24
assert len(kota_tablosu) == 24
assert kota_tablosu.index.is_unique
assert kota_tablosu["Korumalı Kota"].sum() == 400000

# Aynı rastgele örneklerin tekrar seçilebilmesi
rastgele_ureteci = np.random.default_rng(42)

# Sonuçları takip etmek için değişkenler
toplam_secilen = 0
sinif_sayac = Counter()
dosya_sonuclari = []
ilk_yazim = True

try:
    # Her orijinal CSV dosyasını sırayla işleme
    for dosya in ham_dosyalar:

        dosya_adi = dosya.name

        kayit_sayisi = int(
            kota_tablosu.loc[dosya_adi, "Kayıt Sayısı"]
        )

        kota = int(
            kota_tablosu.loc[dosya_adi, "Korumalı Kota"]
        )

        # Dosya içindeki rastgele satır konumlarını seçme
        secilen_indeksler = np.sort(
            rastgele_ureteci.choice(
                kayit_sayisi,
                size=kota,
                replace=False
            )
        )

        dosyadan_secilen = 0
        baslangic = 0

        # Belleği korumak için parçalar halinde okuma
        for parca in pd.read_csv(
            dosya,
            chunksize=20000,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig"
        ):

            bitis = baslangic + len(parca)

            # Bu parçaya düşen seçilmiş satırları bulma
            sol = np.searchsorted(
                secilen_indeksler, baslangic
            )

            sag = np.searchsorted(
                secilen_indeksler, bitis
            )

            yerel_indeksler = (
                secilen_indeksler[sol:sag] - baslangic
            )

            if len(yerel_indeksler) > 0:

                secilen = parca.iloc[
                    yerel_indeksler
                ].copy()

                # Seçilen kayıtları yeni CSV dosyasına ekleme
                secilen.to_csv(
                    gecici_dosya,
                    mode="w" if ilk_yazim else "a",
                    header=ilk_yazim,
                    index=False,
                    encoding="utf-8"
                )

                ilk_yazim = False

                dosyadan_secilen += len(secilen)

                # Saldırı sınıflarının sayılarını tutma
                sinif_sayac.update(
                    secilen["Attack_type"].value_counts().to_dict()
                )

            baslangic = bitis

        # Dosya bazında doğrulama
        assert baslangic == kayit_sayisi, (
            f"{dosya_adi}: Kaynak kayıt sayısı değişmiş!"
        )

        assert dosyadan_secilen == kota, (
            f"{dosya_adi}: Örnekleme kotası uyuşmuyor!"
        )

        toplam_secilen += dosyadan_secilen

        dosya_sonuclari.append({
            "Dosya": dosya_adi,
            "Planlanan": kota,
            "Seçilen": dosyadan_secilen
        })

        print(
            f"{dosya_adi}: "
            f"{dosyadan_secilen:,} kayıt seçildi."
        )

    # Son kontroller
    assert toplam_secilen == 400000
    assert sum(sinif_sayac.values()) == 400000

    # Tamamlanan geçici dosyayı nihai adıyla kaydetme
    gecici_dosya.rename(cikti_dosyasi)

except Exception:
    # Hata durumunda eksik çıktı dosyasını temizleme
    if gecici_dosya.exists():
        gecici_dosya.unlink()
    raise

# Sonuçları gösterme
print("\nÖRNEKLEME TAMAMLANDI")
print("Toplam seçilen kayıt:", toplam_secilen)
print("Kaydedilen dosya:", cikti_dosyasi)

print("\nDosya bazında seçim kontrolü:")
display(pd.DataFrame(dosya_sonuclari))

# Yeni veri setindeki sınıf dağılımı
orneklem_sinif_dagilimi = pd.DataFrame(
    sinif_sayac.items(),
    columns=["Saldırı Türü", "Kayıt Sayısı"]
)

orneklem_sinif_dagilimi["Oran (%)"] = (
    orneklem_sinif_dagilimi["Kayıt Sayısı"]
    / toplam_secilen * 100
).round(4)

print("\nYeni veri setinin sınıf dağılımı:")
display(
    orneklem_sinif_dagilimi.sort_values(
        "Kayıt Sayısı",
        ascending=False
    ).reset_index(drop=True)
)


Backdoor_attack.csv: 474 kayıt seçildi.
DDoS_HTTP_Flood_attack.csv: 4,368 kayıt seçildi.
DDoS_ICMP_Flood_attack.csv: 55,581 kayıt seçildi.
DDoS_TCP_SYN_Flood_attack.csv: 38,527 kayıt seçildi.
DDoS_UDP_Flood_attack.csv: 61,060 kayıt seçildi.
Distance.csv: 21,809 kayıt seçildi.
Flame_Sensor.csv: 20,410 kayıt seçildi.
Heart_Rate.csv: 3,153 kayıt seçildi.
IR_Receiver.csv: 24,941 kayıt seçildi.
MITM_attack.csv: 300 kayıt seçildi.
Modbus.csv: 3,042 kayıt seçildi.
OS_Fingerprinting_attack.csv: 300 kayıt seçildi.
Password_attack.csv: 20,090 kayıt seçildi.
Port_Scanning_attack.csv: 430 kayıt seçildi.
Ransomware_attack.csv: 300 kayıt seçildi.
SQL_injection_attack.csv: 976 kayıt seçildi.
Soil_Moisture.csv: 22,748 kayıt seçildi.
Sound_Sensor.csv: 28,853 kayıt seçildi.
Temperature_and_Humidity.csv: 30,814 kayıt seçildi.
Uploading_attack.csv: 718 kayıt seçildi.
Vulnerability_scanner_attack.csv: 2,782 kayıt seçildi.
Water_Level.csv: 43,775 kayıt seçildi.
XSS_attack.csv: 304 kayıt seçildi.
phValue.csv

,Dosya,Planlanan,Seçilen
0,Backdoor_attack.csv,474,474
1,DDoS_HTTP_Flood_attack.csv,4368,4368
2,DDoS_ICMP_Flood_attack.csv,55581,55581
3,DDoS_TCP_SYN_Flood_attack.csv,38527,38527
4,DDoS_UDP_Flood_attack.csv,61060,61060
5,Distance.csv,21809,21809
6,Flame_Sensor.csv,20410,20410
7,Heart_Rate.csv,3153,3153
8,IR_Receiver.csv,24941,24941
9,MITM_attack.csv,300,300



Yeni veri setinin sınıf dağılımı:


,Saldırı Türü,Kayıt Sayısı,Oran (%)
0,Normal,213790,53.4475
1,DDoS_UDP,61060,15.2650
2,DDoS_ICMP,55581,13.8952
3,DDoS_TCP,38527,9.6318
4,Password,20090,5.0225
5,DDoS_HTTP,4368,1.0920
6,Vulnerability_scanner,2782,0.6955
7,SQL_injection,976,0.2440
8,Uploading,718,0.1795
9,Backdoor,474,0.1185



### 3.5. Çalışma Veri Setinin Oluşturulması — Sonuç ve Yorum

Edge-IIoTset veri setindeki 20.939.622 kayıt arasından, azınlık sınıfı korumalı tabakalı örnekleme yöntemi kullanılarak 400.000 kayıtlık bir çalışma veri seti oluşturulmuştur.

#### Analiz Sonucunda Elde Edilen Bulgular

- Orijinal kayıt sayısı: 20.939.622
- İncelenen orijinal CSV dosyası: 24
- Oluşturulan çalışma veri seti: 400.000 kayıt
- Normal trafik kayıt sayısı: 213.790
- Saldırı trafiği kayıt sayısı: 186.210
- Toplam sınıf sayısı: 15
- Saldırı sınıfı sayısı: 14

**Azınlık Sınıflarının Dağılımı:**

| Saldırı Türü | Seçilen Kayıt |
|---|---:|
| MITM | 300 |
| OS_Fingerprinting | 300 |
| Ransomware | 300 |

#### Uygulanan Yöntem

Örnekleme işleminde önceki aşamada hesaplanan dosya bazlı kotalar kullanılmıştır.

Her CSV dosyasından belirlenen sayıda kayıt, sabit rastgelelik tohumu (`random_state=42`) kullanılarak tekrar seçimsiz rastgele örnekleme ile alınmıştır.

Bellek kullanımını azaltmak amacıyla orijinal CSV dosyaları `chunksize=20000` parametresiyle parçalar halinde okunmuştur.

Seçilen kayıtlar, orijinal CSV dosyaları değiştirilmeden yeni bir çalışma veri setinde birleştirilmiştir.

#### Örnekleme Sonuçlarının Değerlendirilmesi

Örnekleme işlemi sonucunda tüm dosyalardan planlanan sayıda kayıt seçildiği doğrulanmıştır.

Yeni veri setinde normal trafik oranı %53,4475, saldırı trafiği oranı ise %46,5525 olarak hesaplanmıştır.

Böylece genel normal/saldırı dağılımı orijinal veri setindeki oranlara yakın tutulmuştur.

Bununla birlikte MITM, OS_Fingerprinting ve Ransomware gibi azınlık saldırı sınıflarının temsilini artırmak amacıyla bu sınıflara en az 300 kayıt ayrılmıştır.

Bu işlem, azınlık sınıflarının orijinal veri setindeki yüzdelik oranlarını değiştirmiştir.

Bu nedenle oluşturulan alt küme, tüm sınıflar açısından tamamen oransal bir örneklem olarak değerlendirilmemelidir.

#### Sınırlılıklar ve Sonraki Aşamalar

Örnekleme işlemi henüz veri temizleme veya öznitelik dönüşümü içermemektedir.

Seçilen kayıtlarda eksik değer, yinelenen kayıt, veri tipi tutarsızlığı ve aykırı değer kontrolleri sonraki aşamalarda gerçekleştirilecektir.

Ayrıca model eğitiminde veri sızıntısının önlenmesi ve değerlendirme sonuçlarının örnekleme kaynaklı dağılım değişikliklerinden etkilenmesinin dikkate alınması gerekmektedir.

#### Sonuç

Azınlık sınıfı korumalı tabakalı örnekleme yöntemi başarıyla uygulanmış ve 400.000 kayıtlık çalışma veri seti oluşturulmuştur.

Bu veri seti, sonraki veri temizleme, keşifsel veri analizi, öznitelik mühendisliği, öznitelik seçimi ve makine öğrenmesi uygulamalarında kullanılacaktır.



### 3.6. Oluşturulan Veri Setinin Doğrulanması

**Amaç:**

Örnekleme sonucunda oluşturulan CSV dosyasının kayıt sayısını, sütun yapısını ve hedef değişkenlerinin tutarlılığını doğrulamak.

**Yöntem:**

Oluşturulan çalışma veri seti yeniden okunarak toplam kayıt sayısı, sütun sayısı, sınıf dağılımı ve saldırı etiketlerinin tutarlılığı kontrol edilecektir.

Bellek kullanımını azaltmak amacıyla kayıtlar parçalar halinde okunacaktır.

**Beklenen Sonuç:**

- 400.000 kayıt
- 63 sütun
- 15 farklı sınıf
- Attack_label ve Attack_type değişkenleri arasında tutarlılık

Bu aşamada herhangi bir veri değiştirme veya silme işlemi yapılmayacaktır.


In [12]:

import pandas as pd
from pathlib import Path
from collections import Counter

# Oluşturulan çalışma veri setinin yolu
dosya_yolu = Path(
    r"E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri\EdgeIIoT_400000_korumali.csv"
)

# Dosyanın mevcut olduğunu kontrol etme
assert dosya_yolu.is_file(), "Çalışma veri seti bulunamadı!"

# Sadece başlıkları okuyarak sütun sayısını belirleme
sutunlar = pd.read_csv(
    dosya_yolu,
    nrows=0
).columns.tolist()

# Kontrol değişkenleri
toplam_kayit = 0
uyumsuz_etiket = 0
sinif_sayilari = Counter()

# Sadece hedef değişkenleri parçalar halinde okuma
for parca in pd.read_csv(
    dosya_yolu,
    usecols=["Attack_label", "Attack_type"],
    dtype=str,
    keep_default_na=False,
    chunksize=50000
):

    toplam_kayit += len(parca)

    etiket = parca["Attack_label"].str.strip()
    tur = parca["Attack_type"].str.strip()

    # Binary etiket ile sınıf adının uyumunu kontrol etme
    uygun = (
        (etiket.eq("0") & tur.eq("Normal"))
        |
        (
            etiket.eq("1")
            & tur.ne("Normal")
            & tur.ne("")
        )
    )

    uyumsuz_etiket += int((~uygun).sum())

    # Sınıf kayıt sayılarını hesaplama
    sinif_sayilari.update(
        tur.value_counts().to_dict()
    )

# Sonuçlar
print("Toplam kayıt sayısı:", toplam_kayit)
print("Toplam sütun sayısı:", len(sutunlar))
print("Farklı sınıf sayısı:", len(sinif_sayilari))
print("Uyumsuz etiket sayısı:", uyumsuz_etiket)

# Beklenen yapıyı doğrulama
assert toplam_kayit == 400000
assert len(sutunlar) == 63
assert len(sinif_sayilari) == 15
assert uyumsuz_etiket == 0

print("\nVeri seti yapısal doğrulamadan başarıyla geçti!")


Toplam kayıt sayısı: 400000
Toplam sütun sayısı: 63
Farklı sınıf sayısı: 15
Uyumsuz etiket sayısı: 0

Veri seti yapısal doğrulamadan başarıyla geçti!



### 3.6. Oluşturulan Veri Setinin Doğrulanması — Sonuç ve Yorum

Edge-IIoTset veri setinden oluşturulan çalışma alt kümesinin yapısal bütünlüğü ve hedef değişkenlerinin tutarlılığı kontrol edilmiştir.

**Analiz Sonucunda Elde Edilen Bulgular:**

- Toplam kayıt sayısı: 400.000
- Toplam sütun sayısı: 63
- Farklı sınıf sayısı: 15
- Uyumsuz hedef etiketi sayısı: 0

#### Değerlendirme

Örnekleme sonucunda oluşturulan CSV dosyası yeniden okunmuş ve kayıt sayısının planlanan 400.000 değerine eşit olduğu doğrulanmıştır.

Veri setinin 63 sütundan oluştuğu ve 14 saldırı türü ile 1 normal trafik sınıfını içerdiği tespit edilmiştir.

`Attack_label` ve `Attack_type` değişkenleri üzerinde yapılan temel tutarlılık kontrolünde uyumsuz kayıt bulunmamıştır.

Bu kontrol, normal trafik etiketinin 0, saldırı etiketinin ise 1 olması kuralına dayanmaktadır.

#### Sonuç

Oluşturulan 400.000 kayıtlık çalışma veri seti, gerçekleştirilen yapısal doğrulama kontrollerinden başarıyla geçmiştir.

Bununla birlikte yapısal doğrulama, veri setinin tamamen temiz olduğu anlamına gelmemektedir.

Bu nedenle sonraki aşamada eksik değerler, yinelenen kayıtlar, veri tipi tutarsızlıkları ve aykırı değerler incelenecektir.
